# 深度学习计算

## 层和块

In [1]:
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))

X = torch.rand(2, 20)

net(X)

tensor([[ 0.1112, -0.0980, -0.0646, -0.0350, -0.0246, -0.0005, -0.0300,  0.0441,
         -0.0751, -0.1270],
        [-0.0112, -0.1868, -0.0019, -0.0354,  0.0790, -0.0172,  0.0705,  0.0245,
         -0.0706, -0.2046]], grad_fn=<AddmmBackward0>)

### 自定义块

In [2]:
class MLP(nn.Module):
    # 用模型参数声明层。这里，我们声明两个全连接的层
    def __init__(self):
        # 调用MLP的父类Module的构造函数来执行必要的初始化
        # 这样，在类实例化时也可以指定其他函数参数，例如模型参数params
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)

    # 定义模型的前向传播，即如何根据输入X返回所需的模型输出
    def forward(self, X):
        # 注意，这里我们使用ReLU的函数版本，其在nn.functional模块中定义
        return self.out(F.relu(self.hidden(X)))

In [3]:
net = MLP()
net(X)

tensor([[-0.0450,  0.0313,  0.1082, -0.1365, -0.0286,  0.0843, -0.0385,  0.0059,
         -0.0367,  0.1522],
        [-0.1033, -0.0579,  0.1537, -0.2032,  0.0037, -0.0129, -0.0401, -0.1784,
         -0.1574,  0.1078]], grad_fn=<AddmmBackward0>)

### 顺序块

_modules的主要优点是：在模块的参数初始化过程中，系统知道在_modules字典中查找需要初始化参数的⼦块。

In [4]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            # 这里，module是Module子类的一个实例。我们把它把保存在'Module'类的成员
            # 变量_modules中。_module的类型是OrderedDict
            self._modules[str(idx)] = module
    
    def forward(self, X):
        # OrderedDict保证了按照成员添加的顺序遍历它们
        for block in self._modules.values():
            X = block(X)
        return X

In [5]:
net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)

tensor([[ 0.1114, -0.0139,  0.1752, -0.0539, -0.0175,  0.1199, -0.0056,  0.0014,
          0.1970, -0.1634],
        [ 0.2196, -0.0686,  0.1940, -0.0735,  0.0549,  0.1422,  0.0336, -0.0589,
          0.2388, -0.1702]], grad_fn=<AddmmBackward0>)

### 在前向传播函数中执行代码

In [6]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # 不计算梯度的随机权重参数。因此其在训练期间保持不变
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        # 使用创建的常量参数以及relu和mm函数
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        # 复用全连接层。这相当于两个全连接层共享参数
        X = self.linear(X)
        # 控制流
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

In [7]:
net = FixedHiddenMLP()
net(X)

tensor(-0.1963, grad_fn=<SumBackward0>)

In [10]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(),
                                    nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)

    def forward(self, X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
chimera(X)

tensor(0.0898, grad_fn=<SumBackward0>)

## 参数管理

In [1]:
import torch 
from torch import nn

net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
X = torch.rand((2, 4))
net(X)

tensor([[-0.5397],
        [-0.4039]], grad_fn=<AddmmBackward0>)

### 参数管理

In [2]:
print(net[2].state_dict())

OrderedDict([('weight', tensor([[-0.0141, -0.2708, -0.0343, -0.0786, -0.2997,  0.0162, -0.0181, -0.0668]])), ('bias', tensor([-0.0890]))])


#### 目标参数

**参数是复合的对象，包含值、梯度和额外信息。这就是我们需要显式参数值的原因。**

In [3]:
print(type(net[2].bias))
print(net[2].bias)
print(net[2].bias.data)

<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([-0.0890], requires_grad=True)
tensor([-0.0890])


In [4]:
net[2].weight.grad == None

True

#### 一次性访问所有参数

In [6]:
print(*[(name, param.shape) for name, param in net[0].named_parameters()])
print(*[(name, param.shape) for name, param in net.named_parameters()])

('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))


In [7]:
net.state_dict()['2.bias'].data

tensor([-0.0890])

#### 从嵌套快收集参数

In [8]:
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                        nn.Linear(8, 4), nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        net.add_module(f'block {i}', block1())

    return net

rgent = nn.Sequential(block2(), nn.Linear(4, 1))
rgent(X)

tensor([[-0.3214],
        [-0.3215]], grad_fn=<AddmmBackward0>)

In [9]:
print(rgent)

Sequential(
  (0): Sequential(
    (block 0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


In [10]:
rgent[0][1][0].bias.data

tensor([ 0.0799,  0.3366, -0.4735, -0.1699, -0.1036,  0.3813, -0.4816, -0.1367])

### 参数初始化

#### 内置初始化

In [11]:
# 将所有权重初始化为标准差为0.01的高斯随机变量，并将偏执参数设置为0
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean=0, std=0.01)
        nn.init.zeros_(m.bias)
net.apply(init_normal)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([-0.0065, -0.0080,  0.0029,  0.0144]), tensor(0.))

In [13]:
# 还可以将所有参数初始化为给定常数
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 1)
        nn.init.zeros_(m.bias)

net.apply(init_constant)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([1., 1., 1., 1.]), tensor(0.))

In [15]:
# 还可以对某些块应⽤不同的初始化⽅法
def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)
def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 42)

net[0].apply(init_xavier)
net[2].apply(init_42)
net[0].weight.data[0], net[2].weight.data

(tensor([-0.6590,  0.4856,  0.5221, -0.4862]),
 tensor([[42., 42., 42., 42., 42., 42., 42., 42.]]))

#### 自定义初始化

有时，深度学习框架没有提供我们需要的初始化⽅法。在下⾯的例⼦中，我们使⽤以下的分布为任意权重参数w定义初始化⽅法：
$$w \sim \begin{cases}
U(5, 10) & \text{可能性} \frac{1}{4}\\
0 & \text{可能性} \frac{1}{2}\\
U(-10, -5) & \text{可能性} \frac{1}{4}
\end{cases}$$

In [18]:
def my_init(m):
    if type(m) == nn.Linear:
        print("Init", *[(name, param.shape) for name, param in m.named_parameters()][0])
        nn.init.uniform_(m.weight, -10, 10)
        m.weight.data *= m.weight.data.abs() >= 5

net.apply(my_init)
net[0].weight[:2]

Init weight torch.Size([8, 4])
Init weight torch.Size([1, 8])


tensor([[-0.0000, -0.0000, -0.0000, -0.0000],
        [6.3526, 0.0000, 8.5038, -0.0000]], grad_fn=<SliceBackward0>)

In [19]:
# 注意，我们始终可以直接设置参数。
net[0].weight.data[:] += 1
net[0].weight.data[0, 0] = 42
net[0].weight.data[0]

tensor([42.,  1.,  1.,  1.])

### 参数绑定

有时我们希望在多个层间共享参数：我们可以定义⼀个稠密层，然后使⽤它的参数来设置另⼀个层的参数。

In [21]:
# 我们需要给共享层一个名称，以便可以引用它的参数
shared = nn.Linear(8, 8)
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                    shared, nn.ReLU(),
                    shared, nn.ReLU(),
                    nn.Linear(8, 1))
net(X)
# 检查参数是否相同
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0, 0] = 100
# 确保它们实际上是同一个对象，而不是有相同的值
print(net[2].weight.data[0] == net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])


这⾥有⼀个问题：当参数绑定时，梯度会发⽣什么情况？\
答案是由于模型参数包含梯度，因此在反向传播期间第⼆个隐藏层（即第三个神经⽹络层）和第三个隐藏层（即第五个神经⽹络层）的梯度会加在⼀起。

## 延后初始化

### 实例化网络

In [22]:
import torch
from torch import nn

net = nn.Sequential(nn.LazyLinear(256), nn.ReLU(), nn.Linear(256, 10))
print(net)

Sequential(
  (0): LazyLinear(in_features=0, out_features=256, bias=True)
  (1): ReLU()
  (2): Linear(in_features=256, out_features=10, bias=True)
)


In [23]:
[net[i].state_dict() for i in range(len(net))]

[OrderedDict([('weight', <UninitializedParameter>),
              ('bias', <UninitializedParameter>)]),
 OrderedDict(),
 OrderedDict([('weight',
               tensor([[-0.0244, -0.0078,  0.0349,  ...,  0.0609,  0.0107,  0.0594],
                       [ 0.0535, -0.0480,  0.0375,  ...,  0.0030, -0.0610,  0.0004],
                       [-0.0251,  0.0583, -0.0367,  ...,  0.0464, -0.0228, -0.0142],
                       ...,
                       [ 0.0497,  0.0070,  0.0336,  ..., -0.0166,  0.0313,  0.0042],
                       [ 0.0525, -0.0107,  0.0274,  ...,  0.0374, -0.0622,  0.0345],
                       [-0.0130, -0.0503, -0.0377,  ..., -0.0137, -0.0258,  0.0149]])),
              ('bias',
               tensor([ 0.0441,  0.0109, -0.0624,  0.0377,  0.0309,  0.0270,  0.0499,  0.0123,
                        0.0552,  0.0286]))])]

In [25]:
low = torch.finfo(torch.float32).min / 10
high = torch.finfo(torch.float32).max / 10
X = torch.zeros([2, 20], dtype=torch.float32).uniform_(low, high)
net(X)
print(net)

Sequential(
  (0): Linear(in_features=20, out_features=256, bias=True)
  (1): ReLU()
  (2): Linear(in_features=256, out_features=10, bias=True)
)


### 练习

如果输入具有不同的维度，需要做什么？提示：查看参数绑定的相关内容。

In [27]:
# 添加一个额外的线性层，并将第一个线性层的权重与该层的权重绑定在一起。
# 这样就可以解决维度不匹配的问题，并且保持模型的权重不变。
# 注意，在上面的代码中，我们假设第一个线性层的偏置项为零，因此不需要对其进行参数绑定。
import torch
import torch.nn as nn

net = nn.Sequential(
    nn.Linear(20, 256), nn.ReLU(),
    nn.Linear(256, 128), nn.ReLU(),
    nn.Linear(128, 10))

# 添加额外的线性层
extra_layer = nn.Linear(10, 256)

# 将第一个线性层与额外的线性层的权重进行绑定
net[0].weight = extra_layer.weight


# 使用新的输入(维度为20)调用模型
X = torch.rand(2, 10)
net(X)

tensor([[-0.1726, -0.0015,  0.1286, -0.0533, -0.1113,  0.0065, -0.0404, -0.1157,
          0.1334, -0.0488],
        [-0.1791,  0.0165,  0.0953, -0.0488, -0.0807, -0.0064, -0.0355, -0.1066,
          0.1329, -0.0722]], grad_fn=<AddmmBackward0>)

## 自定义层

我们可以通过基本层类设计⾃定义层。这允许我们定义灵活的新层，其⾏为与深度学习框架中的任何现有层不同。

### 不带参数的层

In [1]:
import torch
import torch.nn.functional as F
from torch import nn

class CenteredLayer(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, X):
        return X - X.mean()

In [2]:
layer = CenteredLayer()
layer(torch.FloatTensor([1, 2, 3, 4, 5]))

tensor([-2., -1.,  0.,  1.,  2.])

In [3]:
net = nn.Sequential(nn.Linear(8, 128), CenteredLayer())

In [4]:
Y = net(torch.rand(4, 8))
Y.mean()

tensor(-8.3819e-09, grad_fn=<MeanBackward0>)

### 带参数的层

In [6]:
class MyLinear(nn.Module):
    def __init__(self, in_units, units):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(in_units, units))
        self.bias = nn.Parameter(torch.randn(units,))

    def forward(self, X):
        linear = torch.matmul(X, self.weight.data) + self.bias.data
        return F.relu(linear)

In [7]:
linear = MyLinear(5, 3)
linear.weight

Parameter containing:
tensor([[0.1621, 0.3360, 0.9610],
        [0.6999, 0.3731, 0.2657],
        [0.8187, 0.5945, 0.6038],
        [0.3392, 0.8891, 0.5101],
        [0.2709, 0.2905, 0.5223]], requires_grad=True)

In [8]:
linear(torch.rand(2, 5))

tensor([[1.4542, 0.0000, 1.3232],
        [2.2971, 0.0257, 2.2422]])

In [9]:
net = nn.Sequential(MyLinear(64, 8), MyLinear(8, 1))
net(torch.rand(2, 64))

tensor([[89.3050],
        [85.4010]])

## 读写文件

### 加载和保存张量

In [1]:
import torch
from torch import nn
from torch.nn import functional as F
import os

x = torch.arange(4)
path = os.path.join('..', 'data', 'x-file')
torch.save(x, path)

In [2]:
x2 = torch.load(path)
x2

tensor([0, 1, 2, 3])

In [ ]:
# 我们可以存储⼀个张量列表，然后把它们读回内存。
y = torch.zeros(4)
torch.save([x, y], path)
x2, y2 = torch.load(path)
(x2, y2)

(tensor([0, 1, 2, 3]), tensor([0., 0., 0., 0.]))

In [4]:
# 我们甚⾄可以写⼊或读取从字符串映射到张量的字典。当我们要读取或写⼊模型中的所有权重时，这很⽅便。
mydict = {'x':x, 'y':y}
path = os.path.join('..', 'data', 'mydict')
torch.save(mydict, path)
mydict2 = torch.load(path)
mydict2

{'x': tensor([0, 1, 2, 3]), 'y': tensor([0., 0., 0., 0.])}

### 加载和保存模型参数

需要注意的⼀个重要细节是，这将保存模型的参数⽽不是保存整个模型。因此，为了恢复模型，我们需要⽤代码⽣成架构，然后从磁盘加载参数。

In [5]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.output = nn.Linear(256, 10)

    def forward(self, X):
        return self.output(F.relu(self.hidden(X)))

net = MLP()
X = torch.randn(size = (2, 20))
Y = net(X)

In [6]:
path = os.path.join('..', 'data', 'mlp.params')
torch.save(net.state_dict(), path)

In [7]:
# 读取参数
clone = MLP()
clone.load_state_dict(torch.load(path))
clone.eval()

MLP(
  (hidden): Linear(in_features=20, out_features=256, bias=True)
  (output): Linear(in_features=256, out_features=10, bias=True)
)

In [8]:
Y_clone = clone(X)
Y_clone == Y

tensor([[True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True]])

### 练习

如何同时保存网络架构和参数？需要对架构加上什么限制？

可以使用torch.save()函数同时保存网络架构和参数。为了保存网络架构，需要将模型的结构定义在一个Python类中，并将该类实例化为模型对象。此外，必须**确保该类的构造函数不包含任何随机性质的操作，例如dropout层的随机丢弃概率应该是固定的**

In [3]:
import torch
from torch import nn
from torch.nn import functional as F
import os

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.output = nn.Linear(256, 10)

    def forward(self, X):
        return self.output(F.relu(self.hidden(X)))

net = MLP()

path = os.path.join('..', 'data', 'model.pt')
# 存储模型
torch.save(net.state_dict(), path)

# 导入模型
model = torch.load(path)
model

OrderedDict([('hidden.weight',
              tensor([[ 0.1793, -0.1882,  0.0794,  ..., -0.1649,  0.1484,  0.2068],
                      [ 0.1858, -0.1235, -0.0697,  ..., -0.0556,  0.1920, -0.0113],
                      [-0.0559, -0.1887,  0.0893,  ...,  0.0192, -0.1193,  0.1738],
                      ...,
                      [ 0.2145,  0.0584, -0.1086,  ..., -0.0444,  0.0254, -0.1868],
                      [ 0.1663, -0.0851, -0.0173,  ...,  0.2083, -0.1557,  0.1388],
                      [-0.1526,  0.1216, -0.1222,  ..., -0.0410,  0.1965,  0.1207]])),
             ('hidden.bias',
              tensor([-0.0426,  0.2072,  0.1703, -0.1845,  0.0860, -0.0422, -0.0064,  0.1551,
                      -0.1771,  0.1532,  0.0054, -0.2093, -0.1051, -0.1817, -0.0645,  0.1704,
                       0.0269, -0.1427,  0.1735, -0.0083, -0.2201, -0.0811, -0.1455,  0.0566,
                      -0.1762,  0.0200, -0.1636, -0.0626, -0.0099,  0.1038, -0.1203, -0.1589,
                       0.1374,

## GPU

### 计算设备

应该注意的是，cpu设备意味着所有物理CPU和内存，这意味着PyTorch的计算将尝试使⽤所有CPU核⼼。\
如果有多个GPU，我们使⽤torch.device(f'cuda:{i}') 来表⽰第i块GPU（i从0开始）。\
另外，cuda:0和cuda是等价的。

In [4]:
import torch
from torch import nn

torch.device('cpu'), torch.device('cuda'), torch.device('cuda:1')

(device(type='cpu'), device(type='cuda'), device(type='cuda', index=1))

In [5]:
# 查询可用gpu数量
torch.cuda.device_count()

0

In [6]:
def try_gpu(i = 0): #@save
    """如果存在，则返回gpu(i)，否则返回cpu()"""
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

def try_all_gpus(): #@save
    """返回所有可用的GPU，如果没有GPU，则返回[cpu(),]"""
    devices = [torch.device(f'cuda:{i}') for i in range(torch.cuda.device_count())]
    return devices if devices else [torch.device('cpu')]

try_gpu(), try_gpu(10), try_all_gpus()

(device(type='cpu'), device(type='cpu'), [device(type='cpu')])

### 张量与GPU

我们可以查询张量所在的设备。默认情况下，张量是在CPU上创建的。

In [7]:
x = torch.tensor([1, 2, 3])
x.device

device(type='cpu')

需要注意的是，⽆论何时我们要对多个项进⾏操作，它们都必须在同⼀个设备上。例如，如果我们对两个张量求和，我们需要确保两个张量都位于同⼀个设备上，否则框架将不知道在哪⾥存储结果，甚⾄不知道在哪⾥执⾏计算。

#### 储存在GPU上

In [8]:
X = torch.ones(2, 3, device=try_gpu())
X

tensor([[1., 1., 1.],
        [1., 1., 1.]])

In [9]:
Y = torch.rand(2, 3, device=try_gpu(1))
Y

tensor([[0.2482, 0.7005, 0.3895],
        [0.9764, 0.3114, 0.8662]])

#### 复制

In [ ]:
Z = X.cuda(1)
print(X)
print(Y)
"""
tensor([[1., 1., 1.],
    [1., 1., 1.]], device='cuda:0')
tensor([[1., 1., 1.],
    [1., 1., 1.]], device='cuda:1')
"""

RuntimeError: Cannot access accelerator device when none is available.

In [ ]:
Y + Z
"""
tensor([[1.3821, 1.5270, 1.4919],
    [1.9391, 1.0660, 1.6468]], device='cuda:1')
"""

NameError: name 'Z' is not defined

In [12]:
# 假设变量Z已经存在于第⼆个GPU上。如果我们还是调⽤Z.cuda(1)会发⽣什么？它将返回Z，⽽不会复制并分配新内存。
Z.cuda(1) is Z

NameError: name 'Z' is not defined

### 神经网络与GPU

In [13]:
net = nn.Sequential(nn.Linear(3, 1))
net = net.to(device=try_gpu())

In [ ]:
# 当输⼊为GPU上的张量时，模型将在同⼀GPU上计算结果。
net(X)

tensor([[0.9956],
        [0.9956]], grad_fn=<AddmmBackward0>)

In [15]:
net[0].weight.data.device

device(type='cpu')